In [12]:
"""
surfrad_validation.py
=====================
Technical Validation — r.sun CONUS vs SURFRAD

Produces:
  - Table 1: Overall validation statistics per radiation component
  - Table 2: Per-station validation statistics (all three components)
  - Figure 1: Scatter plot (3-panel)
  - Console output: all inline numbers cited in Section 5.1 and 5.2

Inputs:
  rsun_station_values.csv
  surfrad_processed_0.30/surfrad_multiyear_means.csv

Outputs:
  validation_outputs_0.30/table1_component_stats.csv
  validation_outputs_0.30/table2_station_stats.csv
  validation_outputs_0.30/Fig1_scatter.png

  Elevation values: https://gml.noaa.gov/grad/surfrad/sitepage.html
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
BASE         = r"C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis"
SURFRAD_FILE = os.path.join(BASE, "surfrad_processed_0.30", "surfrad_multiyear_means.csv")
RSUN_FILE    = os.path.join(BASE, "rsun_station_values.csv")
OUT_DIR      = os.path.join(BASE, "validation_outputs_final", "0.30")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family"   : "DejaVu Sans",
    "font.size"     : 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "figure.dpi"    : 150,
})

COMPONENTS = ["glob_rad", "beam_rad", "diff_rad"]
COMPONENT_LABELS = {
    "glob_rad": "Global horizontal (glob_rad)",
    "beam_rad": "Direct beam (beam_rad)",
    "diff_rad": "Diffuse (diff_rad)",
}
STATION_COLORS = {
    "bon": "#1b9e77", "fpk": "#d95f02", "gwn": "#7570b3",
    "tbl": "#e7298a", "dra": "#66a61e", "psu": "#e6ab02", "sxf": "#a6761d",
}
STATION_MARKERS = {
    "bon": "o", "fpk": "^", "gwn": "s",
    "tbl": "D", "dra": "*", "psu": "P", "sxf": "X",
}
STATION_NAMES = {
    "gwn": "Goodwin Creek, MS",
    "dra": "Desert Rock, NV",
    "psu": "Penn State, PA",
    "bon": "Bondville, IL",
    "tbl": "Table Mountain, CO",
    "sxf": "Sioux Falls, SD",
    "fpk": "Fort Peck, MT",
}
STATION_ELEV = {
    "gwn": 98, "dra": 1007, "psu": 376,
    "bon": 230, "tbl": 1689, "sxf": 473, "fpk": 634,
}
STATION_ARIDITY = {
    "dra": "Arid",
    "tbl": "Semi-arid montane",
    "sxf": "Semi-arid",
    "fpk": "Semi-arid",
    "psu": "Humid",
    "bon": "Humid",
    "gwn": "Humid subtropical",
}


In [13]:
# ============================================================
# 1. LOAD AND JOIN DATA
# ============================================================
print("Loading data...")

rsun = pd.read_csv(RSUN_FILE)
rsun = rsun[rsun["doy"] != "annual"].copy()
rsun["doy"]       = rsun["doy"].astype(int)
rsun["rsun_Whm2"] = pd.to_numeric(rsun["rsun_Whm2"], errors="coerce")
rsun = rsun[["station", "doy", "component", "rsun_Whm2"]].copy()

surfrad = pd.read_csv(SURFRAD_FILE)
surfrad["doy"] = surfrad["doy"].astype(int)

surf_mean = surfrad.melt(
    id_vars=["station", "station_name", "lat", "lon", "doy", "n_years"],
    value_vars=["glob_rad_mean", "beam_rad_mean", "diff_rad_mean"],
    var_name="component", value_name="surfrad_Whm2"
)
surf_mean["component"] = surf_mean["component"].str.replace("_mean", "", regex=False)

surf_std = surfrad.melt(
    id_vars=["station", "doy"],
    value_vars=["glob_rad_std", "beam_rad_std", "diff_rad_std"],
    var_name="component", value_name="surfrad_sd"
)
surf_std["component"] = surf_std["component"].str.replace("_std", "", regex=False)

surfrad_long = surf_mean.merge(surf_std, on=["station", "doy", "component"], how="left")

val = rsun.merge(
    surfrad_long[["station", "doy", "component",
                  "surfrad_Whm2", "surfrad_sd",
                  "station_name", "lat", "lon", "n_years"]],
    on=["station", "doy", "component"],
    how="inner"
).dropna(subset=["rsun_Whm2", "surfrad_Whm2"])

val["bias"]      = val["rsun_Whm2"] - val["surfrad_Whm2"]
val["abs_error"] = val["bias"].abs()

print(f"Joined: {len(val)} rows | "
      f"{val['station'].nunique()} stations | "
      f"{val['doy'].nunique()} DOYs | "
      f"{val['component'].nunique()} components")


Loading data...
Joined: 252 rows | 7 stations | 12 DOYs | 3 components


In [4]:
# ============================================================
# 2. COMPUTE STATISTICS
# ============================================================
def compute_stats(group):
    n          = len(group)
    bias       = group["bias"]
    rmse       = np.sqrt(np.mean(bias**2))
    mbe        = np.mean(bias)
    mae        = np.mean(group["abs_error"])
    r2         = np.corrcoef(group["rsun_Whm2"], group["surfrad_Whm2"])[0, 1]**2
    mean_obs   = np.mean(group["surfrad_Whm2"])
    sl, ic, *_ = stats.linregress(group["surfrad_Whm2"], group["rsun_Whm2"])
    return pd.Series({
        "n"         : int(n),
        "R2"        : round(r2,   4),
        "RMSE"      : round(rmse, 1),
        "MBE"       : round(mbe,  1),
        "MAE"       : round(mae,  1),
        "rRMSE_pct" : round(rmse / mean_obs * 100, 2),
        "rMBE_pct"  : round(mbe  / mean_obs * 100, 2),
        "slope"     : round(sl,   4),
        "intercept" : round(ic,   2),
        "mean_obs"  : round(mean_obs, 1),
    })

stats_df      = val.groupby("component",          group_keys=False).apply(compute_stats).reset_index()
station_stats = val.groupby(["station","component"], group_keys=False).apply(compute_stats).reset_index()


In [14]:
# ============================================================
# 3. TABLE 1 — Overall validation statistics per component
# ============================================================
print("\n" + "="*70)
print(" TABLE 1 — Validation statistics per radiation component")
print("="*70)
print(f"{'Component':<34} {'n':>4} {'R2':>6} {'RMSE':>7} {'MBE':>8} "
      f"{'rRMSE%':>8} {'rMBE%':>8}")
print("-"*70)

for comp in COMPONENTS:
    row  = stats_df[stats_df["component"] == comp].iloc[0]
    sign = "+" if row["MBE"] > 0 else ""
    rsgn = "+" if row["rMBE_pct"] > 0 else ""
    print(f"{COMPONENT_LABELS[comp]:<34} "
          f"{row['n']:>4} "
          f"{row['R2']:>6.3f} "
          f"{row['RMSE']:>7.0f} "
          f"{sign}{row['MBE']:>7.0f} "
          f"{row['rRMSE_pct']:>7.1f}% "
          f"{rsgn}{row['rMBE_pct']:>6.1f}%")

table1_rows = [{
    "Component" : COMPONENT_LABELS[c],
    "n"         : stats_df[stats_df["component"]==c].iloc[0]["n"],
    "R2"        : stats_df[stats_df["component"]==c].iloc[0]["R2"],
    "RMSE"      : round(stats_df[stats_df["component"]==c].iloc[0]["RMSE"]),
    "MBE"       : round(stats_df[stats_df["component"]==c].iloc[0]["MBE"]),
    "rRMSE_pct" : stats_df[stats_df["component"]==c].iloc[0]["rRMSE_pct"],
    "rMBE_pct"  : stats_df[stats_df["component"]==c].iloc[0]["rMBE_pct"],
} for c in COMPONENTS]
pd.DataFrame(table1_rows).to_csv(os.path.join(OUT_DIR,"table1_component_stats.csv"), index=False)
print("Saved: table1_component_stats.csv")



 TABLE 1 — Validation statistics per radiation component
Component                             n     R2    RMSE      MBE   rRMSE%    rMBE%
----------------------------------------------------------------------
Global horizontal (glob_rad)       84.0  0.596    2858 +   2407    82.0% +  69.1%
Direct beam (beam_rad)             84.0  0.563    2485 +   2025    85.2% +  69.4%
Diffuse (diff_rad)                 84.0  0.696     403 +    380    71.0% +  67.0%
Saved: table1_component_stats.csv


In [15]:
# ============================================================
# 4. TABLE 2 — Per-station validation statistics
# ============================================================
print("\n" + "="*95)
print(" TABLE 2 — Per-station validation statistics (south to north)")
print("="*95)
print(f"{'Station':<24} {'Lat':>6} {'Elev':>6} "
      f"{'R2glob':>7} {'R2beam':>7} {'R2diff':>7} "
      f"{'MBEglob':>8} {'MBEbeam':>8} {'MBEdiff':>8}")
print("-"*95)

lat_lookup  = val.drop_duplicates("station").set_index("station")["lat"].to_dict()
table2_rows = []

for st in sorted(lat_lookup, key=lambda s: lat_lookup[s]):
    row_dict = {
        "Station"  : STATION_NAMES.get(st, st),
        "Lat (N)"  : round(lat_lookup[st], 2),
        "Elev (m)" : STATION_ELEV.get(st, ""),
    }
    vals_print = []
    for comp in COMPONENTS:
        sub = station_stats[
            (station_stats["station"]   == st) &
            (station_stats["component"] == comp)
        ]
        short = comp.replace("_rad","")
        if len(sub):
            r = sub.iloc[0]
            row_dict[f"R2_{short}"]  = r["R2"]
            row_dict[f"MBE_{short}"] = round(r["MBE"])
            vals_print.extend([r["R2"], round(r["MBE"])])
        else:
            row_dict[f"R2_{short}"]  = None
            row_dict[f"MBE_{short}"] = None
            vals_print.extend([None, None])

    table2_rows.append(row_dict)
    r2g, mbeg, r2b, mbeb, r2d, mbed = vals_print
    print(f"{row_dict['Station']:<24} "
          f"{row_dict['Lat (N)']:>6.2f} "
          f"{str(row_dict['Elev (m)']):>6} "
          f"{r2g:>7.3f} {r2b:>7.3f} {r2d:>7.3f} "
          f"{mbeg:>8} {mbeb:>8} {mbed:>8}")

pd.DataFrame(table2_rows).to_csv(os.path.join(OUT_DIR,"table2_station_stats.csv"), index=False)
print("Saved: table2_station_stats.csv")



 TABLE 2 — Per-station validation statistics (south to north)
Station                     Lat   Elev  R2glob  R2beam  R2diff  MBEglob  MBEbeam  MBEdiff
-----------------------------------------------------------------------------------------------
Goodwin Creek, MS         34.25     98   0.300   0.183   0.815     2404     2086      317
Desert Rock, NV           36.62   1007   0.862   0.856   0.856     1542     1225      317
Bondville, IL             40.05    230   0.515   0.485   0.627     2928     2500      429
Table Mountain, CO        40.12   1689   0.781   0.764   0.839     3040     2565      471
Penn State, PA            40.72    376   0.588   0.562   0.679     2572     2179      392
Sioux Falls, SD           43.73    473   0.833   0.834   0.787     2081     1733      344
Fort Peck, MT             48.31    634   0.796   0.796   0.769     2279     1885      394
Saved: table2_station_stats.csv


In [16]:
# ============================================================
# 5. INLINE NUMBERS — Section 5.2
# ============================================================
print("\n" + "="*70)
print(" INLINE NUMBERS — Section 5.2")
print("="*70)

for comp in COMPONENTS:
    row  = stats_df[stats_df["component"] == comp].iloc[0]
    sign = "+" if row["MBE"] > 0 else ""
    rsgn = "+" if row["rMBE_pct"] > 0 else ""
    print(f"\n{COMPONENT_LABELS[comp]}:")
    print(f"  R2       = {row['R2']:.3f}")
    print(f"  RMSE     = {row['RMSE']:.0f} Wh/m2/day")
    print(f"  MBE      = {sign}{row['MBE']:.0f} Wh/m2/day")
    print(f"  rRMSE    = {row['rRMSE_pct']:.1f}%")
    print(f"  rMBE     = {rsgn}{row['rMBE_pct']:.1f}%")
    print(f"  mean_obs = {row['mean_obs']:.0f} Wh/m2/day")

print("\n--- Per-station glob_rad (aridity gradient, Section 5.2) ---")
for st in sorted(lat_lookup, key=lambda s: lat_lookup[s]):
    sub = station_stats[
        (station_stats["station"]   == st) &
        (station_stats["component"] == "glob_rad")
    ]
    if len(sub):
        r = sub.iloc[0]
        print(f"  {STATION_NAMES.get(st,st):<24}  "
              f"R2={r['R2']:.3f}  MBE={r['MBE']:.0f}  "
              f"Aridity={STATION_ARIDITY.get(st,'')}")



 INLINE NUMBERS — Section 5.2

Global horizontal (glob_rad):
  R2       = 0.596
  RMSE     = 2858 Wh/m2/day
  MBE      = +2407 Wh/m2/day
  rRMSE    = 82.0%
  rMBE     = +69.1%
  mean_obs = 3485 Wh/m2/day

Direct beam (beam_rad):
  R2       = 0.563
  RMSE     = 2485 Wh/m2/day
  MBE      = +2025 Wh/m2/day
  rRMSE    = 85.2%
  rMBE     = +69.4%
  mean_obs = 2917 Wh/m2/day

Diffuse (diff_rad):
  R2       = 0.696
  RMSE     = 403 Wh/m2/day
  MBE      = +380 Wh/m2/day
  rRMSE    = 71.0%
  rMBE     = +67.0%
  mean_obs = 568 Wh/m2/day

--- Per-station glob_rad (aridity gradient, Section 5.2) ---
  Goodwin Creek, MS         R2=0.300  MBE=2404  Aridity=Humid subtropical
  Desert Rock, NV           R2=0.862  MBE=1542  Aridity=Arid
  Bondville, IL             R2=0.515  MBE=2928  Aridity=Humid
  Table Mountain, CO        R2=0.781  MBE=3040  Aridity=Semi-arid montane
  Penn State, PA            R2=0.588  MBE=2572  Aridity=Humid
  Sioux Falls, SD           R2=0.833  MBE=2081  Aridity=Semi-arid
  For

In [17]:
# ============================================================
# 6. FIGURE 1 — Publication-ready scatter plot
#    Same styling as fig1_scatter.py
# ============================================================
print("\nGenerating Figure 1...")

# Publication-ready styling — same as fig1_scatter.py
plt.rcParams.update({
    "font.family"      : "Arial",
    "font.size"        : 9,
    "axes.titlesize"   : 9,
    "axes.labelsize"   : 9,
    "xtick.labelsize"  : 8,
    "ytick.labelsize"  : 8,
    "axes.linewidth"   : 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

# Same figure dimensions as fig1_scatter.py
fig, axes = plt.subplots(
    1, 3,
    figsize=(7.08, 2.8)
)

fig.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.22,
    top=0.93,
    wspace=0.35
)

ax_min = 0

for ax, comp in zip(axes, COMPONENTS):

    # --------------------------------------------------------
    # Data for this radiation component
    # --------------------------------------------------------
    sub = val[val["component"] == comp].copy()

    # Statistics already calculated in stats_df
    row = stats_df[
        stats_df["component"] == comp
    ].iloc[0]

    # --------------------------------------------------------
    # Axis maximum
    # Same logic as fig1_scatter.py
    # --------------------------------------------------------
    ax_max = max(
        sub["rsun_Whm2"].max(),
        sub["surfrad_Whm2"].max()
    ) * 1.08

    # --------------------------------------------------------
    # 1:1 line
    # --------------------------------------------------------
    ax.plot(
        [ax_min, ax_max],
        [ax_min, ax_max],
        color="#999999",
        linestyle="--",
        linewidth=0.8,
        zorder=1,
        label="1:1"
    )

    # --------------------------------------------------------
    # Regression line
    # --------------------------------------------------------
    xline = np.array([ax_min, ax_max])

    ax.plot(
        xline,
        row["slope"] * xline + row["intercept"],
        color="black",
        linewidth=0.9,
        zorder=2,
        label="Regression"
    )

    # --------------------------------------------------------
    # Data points — station by station
    #
    # Code 1 used STATION_STYLE.
    # Code 2 instead has:
    #   STATION_COLORS
    #   STATION_MARKERS
    #   STATION_NAMES
    #
    # So we use those directly.
    # --------------------------------------------------------
    for st in sorted(sub["station"].unique()):

        s = sub[sub["station"] == st]

        marker = STATION_MARKERS.get(st, "o")
        color  = STATION_COLORS.get(st, "gray")
        label  = STATION_NAMES.get(st, st.upper())

        # Same marker-size logic as fig1_scatter.py
        ms = 6 if marker == "*" else 4.5

        ax.scatter(
            s["surfrad_Whm2"],
            s["rsun_Whm2"],
            color=color,
            marker=marker,
            s=ms**2,
            alpha=0.9,
            linewidths=0.3,
            edgecolors="white",
            zorder=3,
            label=label
        )

    # --------------------------------------------------------
    # Statistics annotation
    # Same position/style as fig1_scatter.py
    # --------------------------------------------------------
    sign = "+" if row["MBE"] > 0 else ""

    ann = (
        f"$R^2$ = {row['R2']:.3f}\n"
        f"RMSE = {row['RMSE']:.0f} Wh m$^{{-2}}$ d$^{{-1}}$\n"
        f"MBE = {sign}{row['MBE']:.0f} Wh m$^{{-2}}$ d$^{{-1}}$"
    )

    ax.text(
        0.97,
        0.04,
        ann,
        transform=ax.transAxes,
        fontsize=6,
        va="bottom",
        ha="right",
        linespacing=1.5,
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="#cccccc",
            alpha=0.9
        )
    )

    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------
    ax.set_xlim(ax_min, ax_max)
    ax.set_ylim(ax_min, ax_max)

    ax.set_aspect("equal")

    # Panel title
    panel_title = {
        "glob_rad": "(a) Global horizontal (glob_rad)",
        "beam_rad": "(b) Direct beam (beam_rad)",
        "diff_rad": "(c) Diffuse (diff_rad)",
    }[comp]

    ax.set_title(
        panel_title,
        fontsize=8.5,
        fontweight="bold",
        pad=4
    )

    ax.set_xlabel(
        "SURFRAD measured (Wh m$^{-2}$ day$^{-1}$)",
        fontsize=8
    )

    # Only first panel gets y-axis label
    if ax == axes[0]:
        ax.set_ylabel(
            "r.sun modeled (Wh m$^{-2}$ day$^{-1}$)",
            fontsize=8
        )

    # --------------------------------------------------------
    # Tick formatting — same as fig1_scatter.py
    # --------------------------------------------------------
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f"{x/1000:.0f}k"
            if x >= 1000 else f"{x:.0f}"
        )
    )

    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f"{x/1000:.0f}k"
            if x >= 1000 else f"{x:.0f}"
        )
    )

    ax.tick_params(
        direction="in",
        length=3
    )

    ax.grid(
        True,
        alpha=0.25,
        linewidth=0.5
    )


# ============================================================
# Shared legend below panels
# Same styling as fig1_scatter.py
# ============================================================
from matplotlib.lines import Line2D

legend_handles = [
    Line2D(
        [0],
        [0],
        color="#999999",
        linestyle="--",
        linewidth=0.8,
        label="1:1 line"
    ),

    Line2D(
        [0],
        [0],
        color="black",
        linestyle="-",
        linewidth=0.9,
        label="Regression"
    )
]

# Add the seven SURFRAD stations
for st in STATION_NAMES:

    marker = STATION_MARKERS.get(st, "o")
    color  = STATION_COLORS.get(st, "gray")
    label  = STATION_NAMES.get(st, st.upper())

    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker=marker,
            color=color,
            markersize=5 if marker == "*" else 4,
            linestyle="None",
            label=label
        )
    )

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=5,
    fontsize=7.5,
    frameon=True,
    edgecolor="#cccccc",
    columnspacing=0.8,
    handletextpad=0.4,
    bbox_to_anchor=(0.5, -0.01)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
out_fig = os.path.join(
    OUT_DIR,
    "Fig1_scatter.png"
)

plt.savefig(
    out_fig,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.close()

print(f"Saved: {out_fig}")


Generating Figure 1...
Saved: C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis\validation_outputs_final\0.30\Fig1_scatter.png
